# 05 - Baselines and rolling-origin backtesting

Compare a 12-month seasonal-naive forecast with a recent-three-observation mean. Two origins are used, each with a complete 19-month holdout. Forecasts are generated recursively when a seasonal-naive source falls inside its holdout, so future actuals never leak into predictions.

WAPE is reported overall and by origin, horizon, family, and region. Bias is positive for over-forecasting. Missing targets are excluded from evaluation rather than set to zero.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "outputs" / "features").is_dir():
            return candidate
    raise FileNotFoundError("Run notebooks 01-04 before backtesting.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.backtesting import run_baseline_backtest

summary = run_baseline_backtest(PROJECT_ROOT)
print("Rolling origins:")
for origin in summary["origins"]:
    print(origin)
print(f"Prediction rows: {summary['prediction_rows']:,}")
print(f"Seasonal baseline fallback rows: {summary['fallback_rows_seasonal_naive']:,}")

In [ ]:
import pandas as pd
from IPython.display import display

output_dir = PROJECT_ROOT / "outputs" / "backtesting"
overall = pd.read_csv(output_dir / "accuracy_overall.csv")
by_horizon = pd.read_csv(output_dir / "accuracy_by_horizon.csv")
by_family = pd.read_csv(output_dir / "accuracy_by_family.csv")
by_region = pd.read_csv(output_dir / "accuracy_by_region.csv")
by_family_region = pd.read_csv(output_dir / "accuracy_by_family_region_horizon.csv")
display(overall)
display(by_horizon.head(20))
display(by_family.sort_values("WAPE").head(20))
display(by_region.sort_values("WAPE").head(20))
display(by_family_region.sort_values("WAPE").head(20))

assert set(overall["Model"]) == {"seasonal_naive_12", "recent_mean_3"}
assert set(overall["Origin"]) == {"2023-09", "2025-03"}
assert (overall["Rows"] > 0).all()
print("Validation passed: both baselines evaluated on two full 19-month origins.")

Use this benchmark as the floor for the global model. Compare like-for-like rows and inspect horizon degradation; do not choose a complex model based on fit to training data alone.